In [1]:
%cd /home/qid/MMMM

/home/qid/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")
import time

In [3]:
# Import Pytorch
import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

# Import Pretrain Libraries (transformers + diffusers)
from transformers import BartTokenizer
from diffusers import AutoencoderKL

# Parallel Helper
from parallel import DataParallelModel, DataParallelCriterion

# Import Our Own Functions
from master_init import *
from DSG import *

from count_params import count_params

# Import Tasks
import EEG_TEXT_BART
import EEG_TEXT_BART_SENTIMENT
import EEG_IMG_DIFFUSION
import EEG_IMG_CLASSIFICATION
import PRETRAIN_EEG_IMG_CLIP_MATCHING
import PRETRAIN_EEG_TEXT_CLIP_MATCHING

In [4]:
torch.set_default_dtype(torch.float32)

# config = get_config() Implement later
print(f"[INFO] FETCHING CONFIGURATIONS.")
config = {
    "device" : "cuda",
    "device_ids" : [0,1,2,3],
    "staging_device" : "cuda",
    "num_epochs" : 50,
    "use_non_pytorch_parallel" : False,
    "test_run" : False,
    "live_evaluate" : True,
    "eval_interval" : 1,
    "log_dir" : "./logs"
}

device = config["device"]
device_ids = config["device_ids"]
staging_device = config["staging_device"]
num_epochs = config["num_epochs"]
use_non_pytorch_parallel = config["use_non_pytorch_parallel"]
test_run = config["test_run"]
live_evaluate = config["live_evaluate"]
eval_interval = config["eval_interval"]
log_dir = config["log_dir"]

print(f"[INFO] FINISHED CONFIGURATIONS.")

[INFO] FETCHING CONFIGURATIONS.
[INFO] FINISHED CONFIGURATIONS.


In [5]:
print(f"[INFO] INITIALIZING MODEL.")
model = INITIALIZE_MODEL(device=None, device_ids=device_ids, dtype=torch.float32)
if use_non_pytorch_parallel:
    model = DataParallelModel(model, device_ids=device_ids).to(device)
else:
    model = nn.DataParallel(model, device_ids=device_ids).to(device)
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["ZuCo-BART", "ZuCo-CLIP", "Brain2Image"],
    bsz=[256, 256, 1],
    dev_bsz=[256, 256, 1]
)
if test_run:
    for key, val in dataset_dict.items():
        dataset_dict[key]["train"] = dataset_dict[key]["dev"] # Make things faster

[INFO] INITIALIZING MODEL.
LOADED EEG ENCODER
LOADED CLIP ENCODER
LOADED BART MODEL
LOADED EEG-TEXT-BART
LOADED U-NET
LOADED CLIP TOKENIZER
LOADED EEG-IMG-DIFFUSION
LOADED EEG-IMG-CLASSIFICATION
LOADED EEG-TEXT-SENTIMENT


In [6]:
train_writer = SummaryWriter(log_dir=f"{log_dir}/train-pretrain2")
dev_writer = SummaryWriter(log_dir=f"{log_dir}/dev-pretrain2")

print(f"[INFO] PARAMETER COUNT")
print(f"[INFO] >>>> {count_params(model)} TOTAL PARAMETERS.")
print(f"[INFO] >>>> {count_params(model,True)} TRAINABLE PARAMETERS.")
print(f"[INFO] >>>> {count_params(model,False)} NON-TRAINABLE PARAMETERS.")

# Load Pretrains
print(f"[INFO] LOADING PRETRAINS...")
BART_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(dtype=torch.float32)
vae.requires_grad_(False)
print(f"[INFO] LOADED PRETRAINS.")

# Decision not to put dataloders into DSGTask() object
# Will take too much space and some datasets repeatedly used for
# different tasks.
print(f"[INFO] SETTING UP TRAINING TASKS...")
dsg_tasks = DSGTasks()

dsg_tasks.add_task(
    DSGTask(
        task_name="PRETRAIN-EEG-TEXT-CLIP-MATCHING",
        dataset_tag="ZuCo-CLIP",
        criterion=nn.KLDivLoss(reduction="batchmean"), # Symmetrized with Lambda inside train()
        optimizer=optim.Adam,
        learning_rate=2e-4,
        converge_lim=2,
        converge_threshold=0.005,
        div_threshold=0.01
        )
    )

dsg_tasks.add_task(
    DSGTask(
        task_name="PRETRAIN-EEG-IMG-CLIP-MATCHING",
        dataset_tag="Brain2Image",
        criterion=nn.KLDivLoss(reduction="batchmean"), # Symmetrized with Lambda inside train()
        optimizer=optim.Adam,
        learning_rate=2.5e-5,
        converge_lim=2,
        converge_threshold=0.005,
        div_threshold=0.01
        )
    )

print(f"[INFO] FINISHED SETTING UP TRAINING TASKS.")

[INFO] PARAMETER COUNT
[INFO] >>>> 1508803952 TOTAL PARAMETERS.
[INFO] >>>> 560835244 TRAINABLE PARAMETERS.
[INFO] >>>> 947968708 NON-TRAINABLE PARAMETERS.
[INFO] LOADING PRETRAINS...
[INFO] LOADED PRETRAINS.
[INFO] SETTING UP TRAINING TASKS...
[INFO] FINISHED SETTING UP TRAINING TASKS.


In [9]:
lr_scale = 1.0
lr_scale_cnt = 0
epoch = 0
print(f"[INFO] STARTING TRAINING.")
while epoch < num_epochs:
    print(f"Epoch {epoch}")
    epc_start = time.time()
    for task in dsg_tasks.tasks:
        start = time.time()
        if task.name == "PRETRAIN-EEG-TEXT-CLIP-MATCHING":
            args_dict = {
                "model" : model,
                "dataloader" : dataset_dict[task.dataset_tag],
                "optimizer" : task.optimizer(model.parameters(), lr=task.learning_rate),
                "tokenizer" : BART_tokenizer,
                "criterion" : task.criterion,
                "device" : device,
                "device_ids" : device_ids,
                "staging_device" : staging_device,
                "use_unet" : epoch >= 50,
                "dev_bsz" : 256,
                "bool_eval" : True if live_evaluate and epoch % eval_interval == 0 else False,
                "temperature" : 25
            }
            results = PRETRAIN_EEG_TEXT_CLIP_MATCHING.train(args_dict, using_non_pytorch_parallel=use_non_pytorch_parallel)
            model = results["model"]
            print(f">>>> {task.name} | TRAIN: {results['train_loss']} DEV: {results['dev_loss']} TIME: {time.time() - start:.2f} SECONDS")
            train_writer.add_scalar(f"{task.name} Loss", results['train_loss'], epoch)
            dev_writer.add_scalar(f"{task.name} Loss", results['dev_loss'], epoch)
            task.update()

            if epoch % eval_interval == 0 and live_evaluate:
                print(f">>>>>>>> TRAIN ACCURACY: {results['train_accuracy'] * 100 : 8.4f} % DEV ACCURACY: {results['dev_accuracy'] * 100 : 8.4f} %")
                train_writer.add_scalar(f"{task.name} Accuracy", results['train_accuracy'] * 100, epoch)
                dev_writer.add_scalar(f"{task.name} Accuracy", results['dev_accuracy'] * 100, epoch)


        elif task.name == "PRETRAIN-EEG-IMG-CLIP-MATCHING":
            args_dict = {
                "model" : model,
                "dataloader" : dataset_dict[task.dataset_tag],
                "optimizer" : task.optimizer(model.parameters(), lr=task.learning_rate),
                "criterion" : task.criterion,
                "device" : device,
                "device_ids" : device_ids,
                "staging_device" : staging_device,
                "use_unet" : epoch >= 50,
                "bsz" : 256,
                "bool_eval" : True,
                "temperature" : 25
            }
            results = PRETRAIN_EEG_IMG_CLIP_MATCHING.train(args_dict, using_non_pytorch_parallel=use_non_pytorch_parallel)
#                 print([key for key, val in results.items()])
            model = results["model"]
            print(f">>>> {task.name} | TRAIN: {results['train_loss']} DEV: {results['dev_loss']} TIME: {time.time() - start:.2f} SECONDS")
            train_writer.add_scalar(f"{task.name} Loss", results['train_loss'], epoch)
            dev_writer.add_scalar(f"{task.name} Loss", results['dev_loss'], epoch)
            if epoch % eval_interval == 0 and live_evaluate:
                print(f">>>>>>>> TRAIN ACCURACY: {results['train_accuracy'] * 100 : 8.4f} % DEV ACCURACY: {results['dev_accuracy'] * 100 : 8.4f} %")
                train_writer.add_scalar(f"{task.name} Accuracy", results['train_accuracy'] * 100, epoch)
                dev_writer.add_scalar(f"{task.name} Accuracy", results['dev_accuracy'] * 100, epoch)
        else:
            print(f"[WARNING] Task {task.name} not found. Skipping.")
        try:
            del results
        except:
            pass
    print(f"TOT TIME: {time.time() - epc_start:.2f} SECONDS")
    if epoch % 10 == 0:
        torch.save(model.state_dict(), f"./checkpoints/PretrainPlus/MMMM_{epoch}.pt")
    epoch += 1
torch.save(model.state_dict(), f"./checkpoints/PretrainPlus/MMMM_FINAL.pt")

[INFO] STARTING TRAINING.
Epoch 0
tensor([[0.0038, 0.0039, 0.0039, 0.0044],
        [0.0038, 0.0039, 0.0039, 0.0044],
        [0.0038, 0.0039, 0.0039, 0.0044],
        [0.0038, 0.0039, 0.0039, 0.0044]], device='cuda:0',
       grad_fn=<SliceBackward0>)
tensor([[8.0569e-01, 6.5032e-05, 6.2165e-05, 3.5182e-03],
        [9.5246e-05, 4.2009e-01, 4.5412e-05, 8.4381e-05],
        [7.0945e-07, 3.5386e-07, 9.9978e-01, 4.6580e-06],
        [4.9477e-07, 8.1024e-09, 5.7399e-08, 9.9881e-01]], device='cuda:0')
tensor([[0.0040, 0.0034, 0.0041, 0.0035],
        [0.0040, 0.0034, 0.0041, 0.0035],
        [0.0040, 0.0034, 0.0041, 0.0035],
        [0.0040, 0.0034, 0.0041, 0.0035]], device='cuda:0',
       grad_fn=<SliceBackward0>)
tensor([[9.9868e-01, 1.3095e-07, 2.2751e-07, 7.9585e-08],
        [6.2003e-08, 9.9976e-01, 2.9122e-07, 2.9702e-08],
        [3.4547e-04, 9.3395e-04, 3.0062e-01, 6.4390e-05],
        [2.4996e-09, 1.9702e-09, 1.3318e-09, 1.0000e+00]], device='cuda:0')
tensor([[0.0033, 0.0044, 0.0

KeyboardInterrupt: 

In [ ]:
a = torch.randn((256, 256))

In [ ]:
a-a.max(dim=1).values.unsqueeze(dim=1)

In [ ]:
a = torch.tensor([[3, 6, 100000],[3,6,9]]).to(dtype=torch.float32)

In [ ]:
a-a.max(dim=1).values.unsqueeze(dim=1)

In [ ]:
F.softmax(a)

In [ ]:
F.softmax()

In [ ]:
norm = lambda x : x - x.max(dim=1).values.unsqueeze(dim=1)

In [ ]:
norm(a)

In [10]:
c=nn.CosineEmbeddingLoss()

In [19]:
c(torch.randn((32, 77, 768)).view(32*77,768),torch.randn((32, 77, 768)).view(32*77,768), torch.ones(32*77))

tensor(0.9998)

In [25]:
c(torch.tensor([[1,0]]),torch.tensor([[1,0]]), torch.tensor([1]))

tensor(0.)